# 09m — Estudo de sementes do RF multiclasse no teste (padrão × ajustado)

**Pergunta:** a diferença que vimos no teste entre o RF **padrão** (100 árvores, profundidade livre, folha 1; notebook 06m) e o **ajustado** (100 árvores, profundidade livre, folha 5; notebook 08m) é maior que a variação causada só pelo **sorteio do modelo** (a semente)? Em especial: o `Syn` mandado para `BENIGN` pelo ajustado (40%) e o recall dos ataques (0,725 contra 0,999) mudam com a semente?

O Random Forest sorteia as linhas de cada árvore e as colunas de cada divisão. Com outra semente, o modelo sai um pouco diferente e a nota no teste muda um pouco. Até aqui cada modelo foi treinado com **uma** semente (42). Este notebook treina cada um com várias e mostra a **faixa** de resultados.

| O que | Como |
|---|---|
| **Modelos** | Os mesmos dois conjuntos de hiperparâmetros, com **5 sementes** cada (`SEMENTES_ESTUDO`, padrão 42 a 46). A semente 42 usa os **modelos já gravados** (notebooks 05m e 07m); as outras 4 de cada são **treinadas agora** |
| **Dados de treino** | O mesmo refit do 05m e do 07m (299.589 linhas), lido na mesma ordem do 07m. **Só a semente muda** |
| **Teste** | O mesmo teste do 06m e do 08m (7.216.487 linhas). É lido **uma vez** do servidor e guardado em disco (`cache_teste`, em `float32`, o que não muda as previsões do RF); cada modelo prevê todos os lotes |
| **Métricas** | As do 06m e do 08m (ROC-AUC um-contra-resto, acurácia balanceada, F1 macro com e sem o `UDPLag`, MCC, recall de cada classe) e o desempenho **ataque × BENIGN** (as 6 classes de ataque juntas), o `Syn` previsto como `BENIGN` e como `UDPLag` |
| **Comparação** | Pareada por semente (ajustado − padrão): média, desvio, mínimo, máximo e em quantas sementes o ajustado foi melhor; e se as **faixas das sementes se sobrepõem** |

## O que este estudo é e não é
- **É descritivo.** Mede o ruído do sorteio do modelo e dá uma faixa aos resultados. **Nenhuma escolha depende dele**: nem hiperparâmetros, nem semente.
- **Usa o teste de novo**, só para descrever. Isso deve constar no texto do TCC.
- **Não mede** a variação por usar outras linhas no ajuste ou no refit (os dados de treino são os mesmos).
- A semente 42 do **padrão** é o modelo do 05m, treinado com as linhas do refit em **outra ordem** (a ordem natural da view); as outras sementes usam a ordem do 07m. Isso muda o sorteio das árvores, não os dados.

## Garantias (verificadas por código)
1. **O treino só é permitido dentro de uma função** (`treinar_semente`, seção 3). Qualquer outra chamada de `fit` dá erro.
2. **Nenhum arquivo existente é sobrescrito nem apagado**, nas pastas do 07m e do 05m. Tudo o que este notebook cria é **novo**, dentro de `estudo_sementes` (na pasta da busca do 07m). Cada arquivo só é criado se ainda não existir.
3. **O servidor só é lido** (só `SELECT`).
4. **Conferência final:** compara o tamanho e a data de **todos os arquivos que já existiam** nas duas pastas, antes e depois.
5. **Reprodução:** a semente 42 de cada modelo deve **reproduzir exatamente a matriz de confusão** do 06m e do 08m (confere o cache, o pré-processamento e os modelos). E, por padrão, o notebook **retreina o ajustado com a semente 42** e confere que sai **idêntico** ao modelo gravado pelo 07m (assim se sabe que as sementes novas foram treinadas pelo mesmo procedimento).
6. **Retomada:** cada lote do cache, cada modelo e cada resultado é salvo ao ficar pronto. Se for interrompido, **rode de novo** e ele continua de onde parou.

## O que é criado (na pasta `rfm_busca_<data>\estudo_sementes`)
`cache_teste/` (o teste em disco, **~0,8 a 1,3 GB**; pode ser apagado depois), `modelos/` (os modelos novos), `resultados/` (um arquivo por modelo), `resultados_por_semente.csv`, `resumo_por_modelo.csv`, `diferenca_pareada.csv`, `estudo_sementes.json`, `grafico_sementes.png` e o registro de tempos.

## Antes de rodar
- O 07m e o 08m precisam ter terminado, e o 06m também. O `.env` precisa apontar para o servidor novo.
- **Espaço em disco:** ~1,3 GB para o cache e ~0,4 GB para os modelos novos.
- **Memória:** um modelo por vez (o padrão ocupa ~0,5 GB na memória; o ajustado, bem menos), mais ~0,2 GB das probabilidades do teste. Feche os outros notebooks.
- **Tempo (estimativa minha, sem medir):** ~5 min para ler o teste e guardar; ~7 a 10 min para treinar os 8 modelos (+1 de verificação); ~25 a 35 min para prever e calcular as métricas dos 10 modelos. **Total: ~40 a 55 min.** A barra mostra o andamento.
- Se o computador dormir ou o kernel parar, rode de novo: tudo o que ficou pronto é reaproveitado. **Não mude `LOTE_LINHAS` no meio de uma execução.**


In [1]:
import os, re, json, time, http.client, urllib.parse
from pathlib import Path
import pandas as pd

pd.set_option("display.width", 200); pd.set_option("display.max_rows", 200); pd.set_option("display.max_colwidth", 120)

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()


def load_env(path: Path) -> dict:
    values = {}
    for line in path.read_text().splitlines():
        line = line.strip()
        if not line or line.startswith("#") or "=" not in line:
            continue
        key, _, value = line.partition("=")
        values[key.strip()] = value.strip().strip('"').strip("'")
    return values


env_path = REPO_ROOT / ".env"
if not env_path.exists():
    raise FileNotFoundError(f"Arquivo .env não encontrado em: {env_path}")
env = {**load_env(env_path), **os.environ}

CH_HOST, CH_PORT = env["CLICKHOUSE_HOST"], env["CLICKHOUSE_PORT"]
CH_USER, CH_PASSWORD, CH_DATABASE = env["CLICKHOUSE_USER"], env["CLICKHOUSE_PASSWORD"], env["CLICKHOUSE_DATABASE"]
CH_SCHEME = env.get("CLICKHOUSE_SCHEME", "https")
HTTP_TIMEOUT = 1800
print(f"ClickHouse: {CH_HOST}:{CH_PORT}/{CH_DATABASE}  |  usuário: {CH_USER}")

ClickHouse: clickhouse.afonsolelis.tech:443/railway  |  usuário: clickhouse


In [2]:
# ---------------- transporte HTTP direto (conexão reutilizada + reconexão automática só para leituras) ----------------
class ErroConexao(Exception):
    pass


CONEXAO_ESPERAS = (0, 5, 15, 45, 90)
_conn = None
ULT_STATS = None                  # estatísticas (linhas lidas etc.) da última consulta JSON


def _fechar_conexao():
    global _conn
    try:
        if _conn is not None:
            _conn.close()
    except Exception:
        pass
    _conn = None


def _http(query: str) -> pd.DataFrame:
    """Executa via HTTP. Repetir leituras é seguro; escritas não são reenviadas aqui (quem repete é carregar(), com verificação de contagem)."""
    global _conn, ULT_STATS
    params = {"database": CH_DATABASE, "default_format": "JSONCompact", "wait_end_of_query": 1}
    primeiro = re.match(r"\s*(\w+)", query)
    verbo = primeiro.group(1).upper() if primeiro else ""
    if verbo == "ALTER":
        params["mutations_sync"] = 2          # espera o DELETE terminar antes de responder
    caminho = "/?" + urllib.parse.urlencode(params)
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    somente_leitura = verbo not in {"INSERT", "CREATE", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE", "DELETE", "UPDATE"}
    tentativas = CONEXAO_ESPERAS if somente_leitura else (0,)
    ultimo = None
    for espera in tentativas:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s...")
            time.sleep(espera)
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            corpo = resp.read()
        except (OSError, http.client.HTTPException) as e:
            ultimo = e
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"Falha de conexão durante escrita: {e!r}") from e
            continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} [código ClickHouse: {resp.getheader('X-ClickHouse-Exception-Code')}] corpo={corpo[:300].decode('utf-8', 'replace')!r}")
            _fechar_conexao()
            if not somente_leitura:
                raise ErroConexao(f"HTTP {resp.status} durante escrita. Resposta: {corpo.decode('utf-8', 'replace')[:500]!r}")
            continue
        if resp.status != 200:
            raise Exception(corpo.decode("utf-8", "replace")[:2000])
        if not corpo.strip():
            return pd.DataFrame()
        dados = json.loads(corpo)
        ULT_STATS = dados.get("statistics")
        return pd.DataFrame(dados.get("data", []), columns=[m["name"] for m in dados.get("meta", [])])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(tentativas)} tentativas: {ultimo!r}")

In [3]:
# ---------------- TRAVA: só é possível escrever em tabelas prep_* ----------------
PREFIXO_NOVO = "prep_"          # tabelas que ESTE notebook pode criar / alimentar / apagar

_LITERAL = re.compile(r"'[^'\\]*(?:\\.[^'\\]*)*'")      # literais entre aspas simples (aceita \' escapado)


def _sem_literais(s: str) -> str:
    # texto SQL sem o conteúdo dos literais: ';' ou 'DROP TABLE' dentro de aspas NÃO contam na checagem da trava
    return _LITERAL.sub("''", s)


_VERBOS_ESCRITA = ("CREATE", "INSERT", "DROP", "ALTER", "TRUNCATE", "RENAME", "OPTIMIZE")
_VERBOS_PROIBIDOS = ("DELETE", "UPDATE", "ATTACH", "DETACH", "EXCHANGE", "GRANT", "REVOKE", "KILL", "SYSTEM", "SET", "USE")
_ALVO = re.compile(r"\b(?:TABLE|INTO|VIEW)\s+(?:IF\s+(?:NOT\s+)?EXISTS\s+)?(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)
_RENAME = re.compile(r"(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)\s+TO\s+(`[^`]+`(?:\.`[^`]+`)?|[\w]+(?:\.[\w]+)?)", re.I)


def _tabela_de(ident: str) -> str:
    # 'banco.tabela' ou '`tabela`' -> 'tabela' (e exige que o banco, se informado, seja o atual)
    partes = [p.strip("`") for p in re.findall(r"`[^`]+`|[^.]+", ident)]
    if len(partes) == 2 and partes[0] != CH_DATABASE:
        raise PermissionError(f"Escrita bloqueada: banco '{partes[0]}' diferente do atual '{CH_DATABASE}'.")
    return partes[-1]


def validar_escrita(sql: str, permitir_prefixos=()) -> None:
    # Levanta PermissionError se o comando escreveria fora das tabelas permitidas. Leituras passam sem restrição.
    s = sql.strip().rstrip(";").strip()
    s_chk = _sem_literais(s)
    if ";" in s_chk:
        raise PermissionError("Escrita bloqueada: envie uma instrução por vez.")
    verbo = (re.match(r"\s*(\w+)", s).group(1) if re.match(r"\s*(\w+)", s) else "").upper()
    if verbo in _VERBOS_PROIBIDOS:
        raise PermissionError(f"Comando '{verbo}' não é permitido neste notebook.")
    if verbo not in _VERBOS_ESCRITA:
        return                                                    # SELECT / SHOW / DESCRIBE / EXISTS ...
    prefixos = (PREFIXO_NOVO, *permitir_prefixos)
    if verbo == "RENAME":
        nomes = [_tabela_de(x) for par in _RENAME.findall(s_chk) for x in par]
    else:
        nomes = [_tabela_de(x) for x in _ALVO.findall(s_chk)]
    if not nomes:
        raise PermissionError(f"Escrita bloqueada: não identifiquei a tabela alvo em: {s[:80]}...")
    ruins = [n for n in nomes if not n.startswith(prefixos)]
    if ruins:
        raise PermissionError(f"Escrita bloqueada em {ruins}: só é permitido escrever em tabelas com prefixo {prefixos}.")


def executar(sql: str, permitir_prefixos=()) -> pd.DataFrame:
    validar_escrita(sql, permitir_prefixos)
    return _http(sql)


consultar = executar      # leituras também passam pela trava (e ela as libera)

# autoteste da trava (não toca no servidor)
_deve_passar = ["SELECT 1", "SHOW CREATE TABLE data2_portmap", f"CREATE TABLE {PREFIXO_NOVO}x AS data2_portmap",
                f"INSERT INTO {PREFIXO_NOVO}x SELECT * FROM data2_portmap LIMIT 1", f"DROP TABLE IF EXISTS `{PREFIXO_NOVO}x`",
                f"INSERT INTO {PREFIXO_NOVO}x VALUES ('idêntica a a; teste', 'DROP TABLE data_syn')"]
_deve_bloquear = ["INSERT INTO data2_portmap SELECT 1", "DROP TABLE data2_portmap", "DROP TABLE IF EXISTS data_tftp",
                  "TRUNCATE TABLE data_syn", "ALTER TABLE data_syn DROP COLUMN label", "CREATE TABLE outra AS data_syn",
                  "RENAME TABLE data_syn TO prep_syn", "CREATE DATABASE x", "DELETE FROM data_syn WHERE 1",
                  f"INSERT INTO {PREFIXO_NOVO}x SELECT 1; DROP TABLE data_syn", "SYSTEM DROP MARK CACHE", f"INSERT INTO {PREFIXO_NOVO}x VALUES ('a'); DROP TABLE data_syn"]
for _s in _deve_passar:
    validar_escrita(_s)
for _s in _deve_bloquear:
    try:
        validar_escrita(_s); raise AssertionError(f"a trava deixou passar: {_s}")
    except PermissionError:
        pass
print(f"Trava de segurança OK: {len(_deve_passar)} comandos permitidos e {len(_deve_bloquear)} bloqueados no autoteste.")

Trava de segurança OK: 6 comandos permitidos e 12 bloqueados no autoteste.


In [4]:
# ---------- ferramentas gerais ----------
import os, sys, json, math, time, zlib, platform, threading
from pathlib import Path
from datetime import datetime
import numpy as np
try:
    import psutil
except ImportError:
    psutil = None
    print("⚠ psutil não está instalado: o pico de memória e o hardware não serão registrados. Rode `pip install psutil` e execute esta célula de novo.")
try:
    import sklearn, joblib
    from sklearn.ensemble import RandomForestClassifier
    from sklearn.model_selection import StratifiedKFold
    from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve
except ImportError as e:
    raise ImportError("Faltam bibliotecas: rode `pip install scikit-learn joblib psutil` e execute esta célula de novo.") from e

def lit(s): return "'" + str(s).replace("\\", "\\\\").replace("'", "\\'") + "'"
def q(c): return "`" + str(c).replace("`", "``") + "`"

def consultar_num(sql):
    df = consultar(sql)
    for c in df.columns:
        try: df[c] = pd.to_numeric(df[c])
        except (ValueError, TypeError): pass
    return df

def n_linhas(tab, onde=""):
    return int(consultar_num(f"SELECT count() AS n FROM {tab}" + (f" WHERE {onde}" if onde else "")).n[0])

def existe(tab):
    return int(consultar_num(f"SELECT count() AS n FROM system.tables WHERE database = currentDatabase() AND name = {lit(tab)}").n[0]) > 0

def colunas_view(view):
    return consultar(f"SELECT name FROM system.columns WHERE database = currentDatabase() AND `table` = {lit(view)} ORDER BY position")["name"].tolist()

def ler_view(view, onde=""):
    # Lê uma view inteira. O id_linha (UInt64) fica como texto, para não perder precisão; o resto vira número.
    cols = colunas_view(view)
    sel = ", ".join(("toString(id_linha) AS id_linha" if c == "id_linha" else q(c)) for c in cols)
    df = consultar(f"SELECT {sel} FROM {view}" + (f" WHERE {onde}" if onde else ""))
    for c in df.columns:
        if c != "id_linha": df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def executar_robusto(sql, antes=None, tentativas=5, espera=120):
    # Escritas não são repetidas pelo transporte: se faltar memória no servidor (Code 241) ou a conexão cair, esperamos e refazemos (tudo aqui é idempotente).
    for k in range(1, tentativas + 1):
        try:
            if antes: executar(antes)
            return executar(sql)
        except PermissionError:
            raise
        except Exception as e:
            if isinstance(e, ErroConexao) or "241" in str(e) or "memory limit" in str(e).lower():
                print(f"  ⚠ falha por memória/conexão (tentativa {k}/{tentativas}): {str(e)[:160]}; aguardando {espera}s e refazendo...")
                time.sleep(espera); continue
            raise
    raise RuntimeError(f"não consegui concluir após {tentativas} tentativas: {sql[:100]}...")

def distribuir(total, tamanhos, alpha, tetos):
    # Reparte `total` linhas entre os tipos (alpha = 0: partes iguais), sem passar do teto de cada um; o que sobra de um tipo pequeno vai para os demais.
    q_ = {t: 0 for t in tamanhos}; livres = [t for t in tamanhos if tetos[t] > 0]; restante = int(total)
    if sum(tetos[t] for t in livres) < restante: raise ValueError(f"não há linhas suficientes: pedido {restante:,}, disponível {sum(tetos[t] for t in livres):,}")
    while restante > 0 and livres:
        w = {t: float(tamanhos[t]) ** alpha for t in livres}; sw = sum(w.values())
        alvo = {t: restante * w[t] / sw for t in livres}
        cheios = [t for t in livres if q_[t] + alvo[t] >= tetos[t]]
        if not cheios:
            base = {t: int(alvo[t]) for t in livres}; sobra = restante - sum(base.values())
            for t in sorted(livres, key=lambda t: alvo[t] - base[t], reverse=True)[:sobra]: base[t] += 1
            for t in livres: q_[t] += base[t]
            restante = 0
        else:
            for t in cheios:
                restante -= tetos[t] - q_[t]; q_[t] = tetos[t]; livres.remove(t)
    return q_

def _limpar(o):
    if isinstance(o, dict): return {str(k): _limpar(v) for k, v in o.items()}
    if isinstance(o, (list, tuple)): return [_limpar(v) for v in o]
    if isinstance(o, np.ndarray): return _limpar(o.tolist())
    if isinstance(o, (np.integer,)): return int(o)
    if isinstance(o, (np.floating, float)): return None if (o != o) else float(o)
    if isinstance(o, Path): return str(o)
    return o

def salvar_json(obj, caminho):
    Path(caminho).write_text(json.dumps(_limpar(obj), indent=1, ensure_ascii=False), encoding="utf-8")

### Ferramentas do teste
- **`PreprocessadorRF`**: o pré-processamento do RF (igual ao da view), usado para as classes que o modelo não conhece.
- **`ler_csv_com_espera`**, **`Barra`**, **`fmt_tempo`**: leitura do teste em CSV (com espera se o servidor ficar sem memória) e barra de progresso.
- **`metricas_multi_grande`**: métricas multiclasse para arrays grandes (confusão por contagem), e o intervalo de Wilson.

In [5]:
class PreprocessadorRF:
    # O pré-processamento do RF, igual ao da view definitiva: ±inf e vazios viram "ausente"; nas colunas com indicador, o ausente é trocado pela mediana do
    # TREINO e entra uma coluna <coluna>_ausente (0/1) no fim. A ordem das colunas do modelo é: atributos, depois os indicadores.
    def __init__(self, features, medianas, retiradas=(), versao_views=""):
        self.features, self.medianas, self.retiradas, self.versao_views = list(features), dict(medianas), list(retiradas), versao_views
        self.indicadores = [c for c in self.features if c in self.medianas]
        self.colunas_modelo = self.features + [c + "_ausente" for c in self.indicadores]
        idx = {c: i for i, c in enumerate(self.features)}
        self._pos = np.array([idx[c] for c in self.indicadores], dtype=int); self._med = np.array([self.medianas[c] for c in self.indicadores], dtype=float)
    def transformar(self, X):
        X = np.asarray(X, dtype=np.float64); X = np.where(np.isinf(X), np.nan, X)
        if len(self._pos) == 0: return X
        sub = X[:, self._pos]; aus = np.isnan(sub); X = X.copy(); X[:, self._pos] = np.where(aus, self._med, sub)
        return np.hstack([X, aus.astype(np.float64)])
    def para_dict(self):
        return {"features": self.features, "indicadores": self.indicadores, "colunas_modelo": self.colunas_modelo, "medianas_treino": self.medianas,
                "colunas_retiradas_ablacao": self.retiradas, "versao_views_rf": self.versao_views,
                "regra": "±inf e vazios = ausente; nas colunas com indicador: ausente -> mediana do treino e coluna <coluna>_ausente (0/1) no fim"}

import gc

def rss_mib():
    return psutil.Process().memory_info().rss / 2 ** 20 if psutil else float("nan")

def fmt_tempo(s):
    if s != s or s == float("inf"): return "?"
    s = int(round(s)); h, r = divmod(s, 3600); m, sg = divmod(r, 60)
    return f"{h}h {m:02d}min" if h else (f"{m}min {sg:02d}s" if m else f"{sg}s")

class Barra:
    # Barra de progresso: HTML no Jupyter/VS Code; se não houver IPython, imprime uma linha a cada 5% ou 60 s.
    def __init__(self, total, desc="", intervalo=0.4):
        self.total, self.desc, self.n, self.t0, self.intervalo = max(int(total), 1), desc, 0, time.perf_counter(), intervalo
        self._ult, self._ult_pct, self._extra, self._h = 0.0, -1, "", None
        try:
            from IPython.display import display as _d, HTML as _H
            self._H = _H; self._h = _d(_H(self._html()), display_id=True)
        except Exception:
            self._h = None
    def _info(self):
        dec = time.perf_counter() - self.t0; frac = min(self.n / self.total, 1.0)
        return frac, dec, (dec / self.n * (self.total - self.n) if self.n else float("nan"))
    def _linha(self):
        frac, dec, eta = self._info()
        return f"{self.desc}: {100 * frac:5.1f}% ({self.n}/{self.total}) | decorrido {fmt_tempo(dec)} | restante ~{fmt_tempo(eta)}" + (f" | {self._extra}" if self._extra else "")
    def _html(self):
        frac, dec, eta = self._info()
        return (f"<div style='font-family:monospace;font-size:13px'><b>{self.desc}</b> <progress value='{self.n}' max='{self.total}' style='width:260px;vertical-align:middle'></progress> "
                f"<b>{100 * frac:5.1f}%</b> ({self.n}/{self.total}) &nbsp;|&nbsp; decorrido {fmt_tempo(dec)} &nbsp;|&nbsp; restante ~{fmt_tempo(eta)}"
                + (f"<br><span style='color:#555'>{self._extra}</span>" if self._extra else "") + "</div>")
    def _mostrar(self, forcar=False):
        agora = time.perf_counter()
        if self._h is not None:
            if forcar or agora - self._ult >= self.intervalo: self._h.update(self._H(self._html())); self._ult = agora
        else:
            pct = int(100 * self.n / self.total) // 5
            if forcar or pct != self._ult_pct or agora - self._ult >= 60: print(self._linha(), flush=True); self._ult_pct, self._ult = pct, agora
    def avancar(self, n=1, extra=None):
        self.n += n
        if extra is not None: self._extra = extra
        self._mostrar(forcar=self.n >= self.total)
    def fechar(self, extra=""):
        self._extra = extra or self._extra; self._mostrar(forcar=True)
        if self._h is not None: print(f"{self.desc}: concluído em {fmt_tempo(time.perf_counter() - self.t0)}")


def recall_por_tipo(y, p, g, limiar, n_tipos):
    # Posição 0 = BENIGN (aqui vale a especificidade); posições seguintes = recall de cada tipo de ataque.
    y, g = np.asarray(y), np.asarray(g); pred = np.asarray(p) >= np.float32(limiar)
    pos = np.bincount(g[y == 1], minlength=n_tipos).astype(float); det = np.bincount(g[(y == 1) & pred], minlength=n_tipos).astype(float)
    r = np.divide(det, pos, out=np.full(n_tipos, np.nan), where=pos > 0); neg = float((y == 0).sum()); r[0] = float(((y == 0) & ~pred).sum()) / neg if neg else float("nan")
    return r

class ErroMemoriaServidor(Exception):
    pass


CH_SETTINGS_LEITURA = globals().get("CH_SETTINGS_LEITURA", {"max_threads": 1, "max_block_size": 8192})      # leitura leve para o servidor (menos colunas e blocos abertos ao mesmo tempo)
ESPERAS_MEMORIA = globals().get("ESPERAS_MEMORIA", (30, 60, 120, 240, 480))                            # segundos de espera após "sem memória" (Code 241) no servidor
LIMITE_MEM_SERVIDOR_GIB = globals().get("LIMITE_MEM_SERVIDOR_GIB", 2.0)                                        # volta a tentar antes do fim da espera se a memória do servidor cair abaixo disto


def ler_csv(query, colunas, texto=()):
    # Leitura em FLUXO: o servidor devolve CSV e o pandas lê direto da conexão, sem guardar o texto inteiro nem criar milhões de objetos Python
    # (o caminho JSON do consultar() gasta várias vezes mais memória). Mesmas regras de repetição do _http: só SELECT, com novas tentativas.
    global _conn
    if not re.match(r"\s*SELECT\b", query, re.I): raise PermissionError("ler_csv só executa SELECT")
    caminho = "/?" + urllib.parse.urlencode({"database": CH_DATABASE, "default_format": "CSV", "wait_end_of_query": 1, **CH_SETTINGS_LEITURA})
    cab = {"X-ClickHouse-User": CH_USER, "X-ClickHouse-Key": CH_PASSWORD, "Connection": "keep-alive"}
    ultimo = None
    for espera in CONEXAO_ESPERAS:
        if espera:
            print(f"  conexão instável ({type(ultimo).__name__}: {str(ultimo)[:160]}); nova tentativa em {espera}s..."); time.sleep(espera)
        corpo = b""
        try:
            if _conn is None:
                cls = http.client.HTTPSConnection if CH_SCHEME == "https" else http.client.HTTPConnection
                _conn = cls(CH_HOST, int(CH_PORT), timeout=HTTP_TIMEOUT)
            _conn.request("POST", caminho, body=query.encode("utf-8"), headers=cab)
            resp = _conn.getresponse()
            if resp.status == 200:
                try:
                    return pd.read_csv(resp, header=None, names=colunas, dtype={c: (str if c in texto else np.float64) for c in colunas}, na_values=["\\N"])
                except pd.errors.EmptyDataError:
                    return pd.DataFrame({c: pd.Series(dtype=(object if c in texto else np.float64)) for c in colunas})
            corpo = resp.read()
        except (OSError, http.client.HTTPException, pd.errors.ParserError, ValueError) as e:
            ultimo = e; _fechar_conexao(); continue
        if resp.status in (502, 503, 504) or (resp.status >= 500 and not corpo.startswith(b"Code:")):
            ultimo = Exception(f"HTTP {resp.status} corpo={corpo[:300].decode('utf-8', 'replace')!r}"); _fechar_conexao(); continue
        msg = corpo.decode("utf-8", "replace")
        if "MEMORY_LIMIT_EXCEEDED" in msg or msg.startswith("Code: 241"):
            raise ErroMemoriaServidor(msg[:600])
        raise Exception(msg[:2000])
    raise ErroConexao(f"Sem conexão com o ClickHouse após {len(CONEXAO_ESPERAS)} tentativas: {ultimo!r}")


def _mem_servidor_gib():
    try:
        return float(consultar_num("SELECT toFloat64(value) AS v FROM system.asynchronous_metrics WHERE metric = 'MemoryResident'")["v"][0]) / 2 ** 30
    except Exception:
        return None


def ler_csv_com_espera(query, colunas, texto=()):
    # Se o servidor ficar sem memória (Code 241, comum num servidor de 3 GiB com merges em andamento), espera e tenta de novo; o erro é transitório.
    for k, espera in enumerate(list(ESPERAS_MEMORIA) + [None]):
        try:
            return ler_csv(query, colunas, texto)
        except ErroMemoriaServidor as e:
            if espera is None:
                raise ErroMemoriaServidor("O servidor continua sem memória depois de todas as esperas. Rode a célula de novo mais tarde: os lotes já prontos são reaproveitados. Último erro: " + str(e)[:300]) from None
            m = _mem_servidor_gib()
            print(f"  ⚠ o servidor ficou sem memória (Code 241; memória do servidor: {'?' if m is None else f'{m:.2f} GiB'}); esperando até {espera}s e tentando de novo ({k + 1}/{len(ESPERAS_MEMORIA)})...", flush=True)
            t0 = time.time()
            while time.time() - t0 < espera:
                time.sleep(min(15, espera))
                m = _mem_servidor_gib()
                if m is not None and m < LIMITE_MEM_SERVIDOR_GIB: break

_b = Barra(2, "autoteste"); _b.avancar(2); assert recall_por_tipo(np.array([0, 1, 1]), np.array([.1, .9, .2]), np.array([0, 1, 1]), 0.5, 2)[1] == 0.5
print("Autoteste das ferramentas do teste OK")

# ---------- métricas multiclasse para arrays grandes (sem cópias em float64; confusão por contagem) ----------
def metricas_multi_grande(y, P):
    y = np.asarray(y, dtype=np.int64); K_ = P.shape[1]; pred = P.argmax(axis=1)
    cm = np.bincount(y * K_ + pred, minlength=K_ * K_).reshape(K_, K_).astype(np.float64)
    s, t, p, d = cm.sum(), cm.sum(axis=1), cm.sum(axis=0), np.diag(cm)
    rec = np.divide(d, t, out=np.full(K_, np.nan), where=t > 0); prec = np.divide(d, p, out=np.zeros(K_), where=p > 0); f1 = np.divide(2 * d, t + p, out=np.zeros(K_), where=(t + p) > 0)
    den = math.sqrt(max((s * s - (p * p).sum()) * (s * s - (t * t).sum()), 0.0)); mcc = float((d.sum() * s - (p * t).sum()) / den) if den > 0 else 0.0
    auc_k = np.full(K_, np.nan)
    for k in range(K_):
        yk = (y == k)
        if 0 < yk.sum() < len(yk): auc_k[k] = roc_auc_score(yk, np.ascontiguousarray(P[:, k], dtype=np.float64))
    esc = {"acuracia": float(d.sum() / s), "acuracia_balanceada": float(np.nanmean(rec)), "precisao_macro": float(prec.mean()), "f1_macro": float(f1.mean()), "f1_ponderado": float((f1 * t).sum() / t.sum()), "mcc": mcc, "roc_auc_ovr_macro": float(np.nanmean(auc_k))}
    return esc, {"cm": cm, "recall": rec, "precisao": prec, "f1": f1, "auc": auc_k, "suporte": t, "previstas": p}

def wilson(k, n, z=1.96):
    if n == 0: return (np.nan, np.nan)
    ph = k / n; c = (ph + z * z / (2 * n)) / (1 + z * z / n); h = z * math.sqrt(ph * (1 - ph) / n + z * z / (4 * n * n)) / (1 + z * z / n); return (c - h, c + h)

# autoteste (não toca no servidor): confere contra o scikit-learn
from sklearn.metrics import accuracy_score as _acc, balanced_accuracy_score as _bacc, f1_score as _f1, matthews_corrcoef as _mcc
_r = np.random.default_rng(1); _y = _r.integers(0, 4, 500); _P = _r.dirichlet(np.ones(4), 500); _P[np.arange(500), _y] += 0.3; _P = (_P / _P.sum(axis=1, keepdims=True)).astype(np.float32); _pr = _P.argmax(axis=1)
_e, _a = metricas_multi_grande(_y, _P)
assert abs(_e["acuracia"] - _acc(_y, _pr)) < 1e-12 and abs(_e["acuracia_balanceada"] - _bacc(_y, _pr)) < 1e-12 and abs(_e["f1_macro"] - _f1(_y, _pr, average="macro")) < 1e-12 and abs(_e["mcc"] - _mcc(_y, _pr)) < 1e-12, _e
assert abs(_e["roc_auc_ovr_macro"] - roc_auc_score(_y, _P.astype(np.float64), multi_class="ovr", average="macro")) < 1e-9
_lo, _hi = wilson(50, 100); assert 0.40 < _lo < 0.41 and 0.59 < _hi < 0.60
print("Autoteste das ferramentas do teste multiclasse OK")

# ---------- proteções: o que este notebook NUNCA faz (sobrescrever, apagar, treinar fora de treinar_semente) ----------
import warnings, shutil
def silenciar_aviso_sklearn():
    # Com muitos núcleos, o scikit-learn às vezes deixa a lista de filtros de aviso VAZIA (condição de corrida entre threads) e passa a emitir, a cada árvore, um aviso inofensivo
    # sobre `delayed`/`Parallel`. Este desvio descarta SÓ essa mensagem; os outros avisos continuam funcionando. Não muda nenhum resultado.
    warnings.filterwarnings("ignore", message=".*sklearn.utils.parallel.delayed.*", category=UserWarning)
    if not getattr(warnings.warn, "_filtrado_sklearn", False):
        _orig = warnings.warn
        def _warn_filtrado(message, *args, **kwargs):
            if isinstance(message, str) and message.startswith("`sklearn.utils.parallel.delayed` should be used with"): return
            return _orig(message, *args, **kwargs)
        _warn_filtrado._filtrado_sklearn = True; warnings.warn = _warn_filtrado
silenciar_aviso_sklearn()

TREINOS_BLOQUEADOS, TREINO_PERMITIDO, RAIZES_TRAVADAS, CRIADOS, MANTIDOS = [], [False], [], [], []
def instalar_travas(pasta, permitir_regravar=()):
    # (1) o treino do RF só é permitido quando TREINO_PERMITIDO[0] é True (só dentro de treinar_semente); (2) bloqueia gravação que SOBRESCREVA arquivo existente dentro das pastas
    # protegidas, e qualquer remoção de arquivo dentro delas. Pode ser chamada para várias pastas; só arquivos temporários (.tmp) e acréscimos (modo "a") podem ser regravados.
    from sklearn.ensemble import RandomForestClassifier as _RF
    import numpy as _np, pandas as _pd, joblib as _jl, pathlib as _pl
    RAIZES_TRAVADAS.append(str(Path(pasta).resolve())); perm = set(permitir_regravar)
    if getattr(_RF.fit, "_bloqueado", False): return
    def dentro(alvo):
        try: s_ = str(Path(alvo).resolve())
        except Exception: return False
        return any(s_.startswith(r) for r in RAIZES_TRAVADAS)
    def checa(alvo, acao, permitir_acrescimo=False):
        if not isinstance(alvo, (str, os.PathLike)) or not dentro(alvo): return
        p = Path(alvo).resolve()
        if p.exists() and p.name not in perm and ".tmp" not in p.name and not permitir_acrescimo:
            raise PermissionError(f"BLOQUEADO: {acao} sobrescreveria '{p.name}', que já existe. Este notebook só cria arquivos novos.")
    def embrulha(obj, nome, pos, acao, kw="", acrescimo_ok=False):
        orig = getattr(obj, nome)
        if getattr(orig, "_travado", False): return
        def f(*a, **k):
            alvo = a[pos] if len(a) > pos else k.get(kw)
            checa(alvo, acao, permitir_acrescimo=(acrescimo_ok and k.get("mode") == "a")); return orig(*a, **k)
        f._travado = True; f.__name__ = getattr(orig, "__name__", nome); setattr(obj, nome, f)
    embrulha(_pd.DataFrame, "to_csv", 1, "gravar CSV", "path_or_buf", acrescimo_ok=True); embrulha(_np, "savez", 0, "gravar NPZ", "file"); embrulha(_np, "savez_compressed", 0, "gravar NPZ", "file")
    embrulha(_jl, "dump", 1, "gravar modelo", "filename"); embrulha(_pl.Path, "write_text", 0, "gravar arquivo"); embrulha(_pl.Path, "write_bytes", 0, "gravar arquivo")
    embrulha(os, "replace", 1, "substituir arquivo"); embrulha(os, "rename", 1, "renomear arquivo")
    try:
        from matplotlib.figure import Figure; embrulha(Figure, "savefig", 1, "gravar figura", "fname")
    except ImportError: pass
    def _proibe_remocao(orig):
        def f(*a, **k):
            alvo = a[0] if a else None
            if isinstance(alvo, (str, os.PathLike)) and dentro(alvo): raise PermissionError(f"BLOQUEADO: apagar '{Path(alvo).name}'. Este notebook nunca apaga arquivos.")
            return orig(*a, **k)
        f._travado = True; return f
    if not getattr(os.remove, "_travado", False): os.remove = _proibe_remocao(os.remove)
    if not getattr(_pl.Path.unlink, "_travado", False):
        _unl = _pl.Path.unlink
        def _u(self, *a, **k):
            if dentro(self): raise PermissionError(f"BLOQUEADO: apagar '{self.name}'. Este notebook nunca apaga arquivos.")
            return _unl(self, *a, **k)
        _u._travado = True; _pl.Path.unlink = _u
    _fit_original = _RF.fit
    def _fit_bloqueado(self, *a, **k):
        if TREINO_PERMITIDO[0]: return _fit_original(self, *a, **k)
        TREINOS_BLOQUEADOS.append(1); raise RuntimeError("BLOQUEADO: treinar um RF só é permitido dentro da função treinar_semente (seção 3).")
    _fit_bloqueado._bloqueado = True; _RF.fit = _fit_bloqueado

def gravar(nome, escrever):
    # Cria o arquivo (relativo a estudo_sementes) SÓ se ele ainda NÃO existir (nunca sobrescreve). Registra o que foi criado e o que foi mantido.
    caminho = PASTA_EST / nome
    if caminho.exists(): MANTIDOS.append(nome); return False
    escrever(caminho); CRIADOS.append(nome); return True
def gravar_csv(df, nome, **kw): return gravar(nome, lambda p: df.to_csv(p, **kw))
def gravar_json(obj, nome): return gravar(nome, lambda p: salvar_json(obj, p))
def tmp_novo(caminho): return caminho.with_name(caminho.stem + ".tmp" + caminho.suffix)

def colapsa(cm_, i_syn):
    # ataque × BENIGN (as 6 classes de ataque juntas; BENIGN é a classe 0) e o destino do Syn
    cm_ = np.asarray(cm_, float); tn, fp, fn, tp = cm_[0, 0], cm_[0, 1:].sum(), cm_[1:, 0].sum(), cm_[1:, 1:].sum(); den = math.sqrt((tp + fp) * (tp + fn) * (tn + fp) * (tn + fn))
    return {"recall dos ataques": tp / (tp + fn), "falso alarme": fp / (fp + tn), "MCC ataque × BENIGN": (tp * tn - fp * fn) / den if den > 0 else 0.0, "ataques previstos como BENIGN": int(fn)}


Autoteste das ferramentas do teste OK
Autoteste das ferramentas do teste multiclasse OK


## 1. Parâmetros e carga
Localiza as duas pastas de modelos (o padrão do 05m/06m e o ajustado do 07m/08m), confere que usam os **mesmos atributos e medianas**, ativa as proteções e tira a fotografia dos arquivos existentes.

In [6]:
# ---------- 1.1 Parâmetros e carga ----------
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score
SEMENTES_ESTUDO = tuple(globals().get("SEMENTES_ESTUDO", (42, 43, 44, 45, 46)))
LOTE_LINHAS = globals().get("LOTE_LINHAS", 50_000)                  # NÃO mude no meio de uma execução (o cache do teste usa esse tamanho)
COMPRESSAO_MODELO = globals().get("COMPRESSAO_MODELO", 3); VERIFICAR_REPRODUCAO = globals().get("VERIFICAR_REPRODUCAO", True)
assert 42 in SEMENTES_ESTUDO and len(set(SEMENTES_ESTUDO)) == len(SEMENTES_ESTUDO) >= 2, "SEMENTES_ESTUDO precisa incluir a 42 (os modelos já gravados) e ter pelo menos 2 sementes diferentes"
BASE_ART = Path(globals().get("REPO_ROOT", Path.cwd())) / "artefatos_modelos"; PASTA_BASE = BASE_ART / "rf_multiclasse_busca"
PASTA_RUN = globals().get("PASTA_RUN", None)
if PASTA_RUN is None:
    runs = sorted(p for p in PASTA_BASE.glob("rfm_busca_*") if (p / "modelo_final_refit.joblib").exists() and (p / "FINALIZADO").exists() and (p / "metricas_teste.json").exists())
    assert runs, "nenhuma pasta rfm_busca_* FINALIZADA com modelo final e teste (08m) em " + str(PASTA_BASE) + ": rode antes os notebooks 07m e 08m (ou defina PASTA_RUN)"
    if len(runs) > 1: print("⚠ há mais de uma busca finalizada; usando a mais recente. Para escolher outra, defina PASTA_RUN:", [p.name for p in runs])
    PASTA_RUN = runs[-1]
PASTA_RUN = Path(PASTA_RUN); assert (PASTA_RUN / "FINALIZADO").exists() and (PASTA_RUN / "metricas_teste.json").exists(), f"{PASTA_RUN.name} precisa estar FINALIZADA e já ter o teste do 08m"
PASTA_PADRAO = globals().get("PASTA_PADRAO", None)
if PASTA_PADRAO is None:
    _c = sorted(p for p in (BASE_ART / "rf_multiclasse").glob("rfm_*") if all((p / f).exists() for f in ("modelo_final_refit.joblib", "refit_ids.csv", "config_treino.json", "preprocessamento.json", "metricas_teste.json")))
    assert _c, "nenhuma pasta rfm_* do 05m com o modelo padrão, o refit_ids.csv e o teste (06m) em " + str(BASE_ART / "rf_multiclasse"); PASTA_PADRAO = _c[-1]
PASTA_PADRAO = Path(PASTA_PADRAO); PASTA_EST = PASTA_RUN / "estudo_sementes"
VIEW_TESTE, V_REFIT, TAB_CLASSES, TAB_TESTE = "prep_vw_rf_multi_teste_sem_sobrep", "prep_vw_bal_multi_refit", "prep_classes_rf_multi", "prep_final_teste"
assert existe(VIEW_TESTE) and existe(V_REFIT) and existe(TAB_CLASSES), f"faltam {VIEW_TESTE}, {V_REFIT} ou {TAB_CLASSES}: rode os notebooks 03m, 04m e 05m"
# fotografia dos arquivos que JÁ EXISTEM (fora de estudo_sementes) nas duas pastas, e proteções
SNAP_ANTES = {("busca/" + str(f.relative_to(PASTA_RUN))): (f.stat().st_size, f.stat().st_mtime_ns) for f in PASTA_RUN.rglob("*") if f.is_file() and PASTA_EST not in f.parents}
SNAP_ANTES.update({("padrao/" + str(f.relative_to(PASTA_PADRAO))): (f.stat().st_size, f.stat().st_mtime_ns) for f in PASTA_PADRAO.rglob("*") if f.is_file()})
PASTA_EST.mkdir(exist_ok=True); instalar_travas(PASTA_RUN); instalar_travas(PASTA_PADRAO); silenciar_aviso_sklearn()
print(f"🔒 Proteções ativas: treino de RF só dentro de treinar_semente; sobrescrita e remoção BLOQUEADAS nas pastas {PASTA_RUN.name} e {PASTA_PADRAO.name} ({len(SNAP_ANTES)} arquivos já existem e serão conferidos no final); o servidor só é lido.")

pc, pc_pad = (json.loads((p / "preprocessamento.json").read_text(encoding="utf-8")) for p in (PASTA_RUN, PASTA_PADRAO))
assert pc["colunas_modelo"] == pc_pad["colunas_modelo"] and pc["medianas_treino"] == pc_pad["medianas_treino"], "o padrão (05m) e o ajustado (07m) não usam os mesmos atributos/medianas"
cfg_aj, cfg_pad = (json.loads((p / "config_treino.json").read_text(encoding="utf-8")) for p in (PASTA_RUN, PASTA_PADRAO))
HP = {"padrao": dict(cfg_pad["hiperparametros"]), "ajustado": dict(cfg_aj["hiperparametros"])}; ROT = {"padrao": "padrão", "ajustado": "ajustado"}
assert HP["padrao"]["random_state"] == 42 and HP["ajustado"]["random_state"] == 42, "os modelos gravados deveriam ter sido treinados com a semente 42"
assert HP["padrao"]["n_estimators"] == HP["ajustado"]["n_estimators"] and HP["padrao"]["min_samples_leaf"] != HP["ajustado"]["min_samples_leaf"]
cls_gravado = json.loads((PASTA_RUN / "classes.json").read_text(encoding="utf-8")); CLASSES, COD = cls_gravado["classes"], cls_gravado["codigos"]; K = len(CLASSES); assert [COD[c] for c in CLASSES] == list(range(K)) and COD["BENIGN"] == 0
prep = PreprocessadorRF(pc["features"], pc["medianas_treino"], pc["colunas_retiradas_ablacao"], pc["versao_views_rf"]); CM = prep.colunas_modelo; assert prep.colunas_modelo == pc["colunas_modelo"]
assert colunas_view(VIEW_TESTE)[:2] == ["id_linha", "y"] and colunas_view(VIEW_TESTE)[2:] == CM and [c for c in colunas_view(V_REFIT) if c not in ("id_linha", "y")] == CM, "as colunas das views não coincidem com as do modelo"
cl = consultar_num(f"SELECT codigo, classe, linhas_teste_sem_sobrep FROM {TAB_CLASSES} ORDER BY codigo"); assert cl.classe.tolist() == CLASSES
N_CLASSE = {c: int(n) for c, n in zip(cl.classe, cl.linhas_teste_sem_sobrep)}; N_TESTE = n_linhas(VIEW_TESTE); assert sum(N_CLASSE.values()) == N_TESTE
ARQ_42 = {"padrao": PASTA_PADRAO / "modelo_final_refit.joblib", "ajustado": PASTA_RUN / "modelo_final_refit.joblib"}
print(f"Padrão (05m/06m): {PASTA_PADRAO.name} | min_samples_leaf = {HP['padrao']['min_samples_leaf']} | Ajustado (07m/08m): {PASTA_RUN.name} | min_samples_leaf = {HP['ajustado']['min_samples_leaf']} | {HP['ajustado']['n_estimators']} árvores, profundidade {HP['ajustado']['max_depth']}, class_weight = {HP['ajustado']['class_weight']}")
print(f"Sementes do estudo: {SEMENTES_ESTUDO} (a 42 usa os modelos já gravados) | Teste: {N_TESTE:,} linhas | atributos: {len(CM)}")
T_SEC = {}


🔒 Proteções ativas: treino de RF só dentro de treinar_semente; sobrescrita e remoção BLOQUEADAS nas pastas rfm_busca_20261102_001711 e rfm_20261101_210334 (396 arquivos já existem e serão conferidos no final); o servidor só é lido.
Padrão (05m/06m): rfm_20261101_210334 | min_samples_leaf = 1 | Ajustado (07m/08m): rfm_busca_20261102_001711 | min_samples_leaf = 5 | 100 árvores, profundidade None, class_weight = balanced
Sementes do estudo: (42, 43, 44, 45, 46) (a 42 usa os modelos já gravados) | Teste: 7,216,487 linhas | atributos: 44


## 2. Cache do teste em disco
Lê o teste **uma única vez** do servidor, em lotes, e guarda cada lote em disco (`float32`; o RF converte para `float32` de qualquer jeito, então as previsões não mudam). Cada modelo prevê os lotes do disco, um de cada vez, sem reler o servidor. **Retomável:** os lotes prontos são reaproveitados.

In [7]:
# ---------- 2.1 Cache do teste em disco (uma única leitura do servidor) ----------
t_ini = time.perf_counter(); PASTA_CACHE = PASTA_EST / "cache_teste"; PASTA_CACHE.mkdir(exist_ok=True)
B = max(1, math.ceil(N_TESTE / LOTE_LINHAS)); cols_sql = ", ".join("v." + q(c) for c in CM); barra = Barra(B, "Cache do teste"); novos = 0
for j in range(B):
    arq = PASTA_CACHE / f"lote_{j:03d}_de_{B:03d}.npz"
    if arq.exists(): barra.avancar(1, f"lote {j + 1}/{B}: já pronto"); continue
    lo, hi = j * 2 ** 64 // B, (j + 1) * 2 ** 64 // B; onde_v = f"v.id_linha >= {lo}" + (f" AND v.id_linha < {hi}" if j < B - 1 else "")
    barra.avancar(0, f"lote {j + 1}/{B}: lendo... (memória do processo {rss_mib():,.0f} MiB)")
    df = ler_csv_com_espera(f"SELECT toString(v.id_linha) AS id, v.y, {cols_sql} FROM {VIEW_TESTE} AS v WHERE {onde_v}", ["id", "y"] + CM, texto=("id",))
    X, y, ids = df[CM].to_numpy(float), df["y"].to_numpy(np.uint8), df["id"].astype("uint64").to_numpy()
    if len(y): assert np.isfinite(X).all() and int(y.max()) < K, "valores não finitos ou classe fora do modelo no teste"
    tmp = tmp_novo(arq); np.savez_compressed(tmp, id=ids, y=y, X=X.astype(np.float32)); os.replace(tmp, arq); novos += 1
    barra.avancar(1, f"lote {j + 1}/{B}: {len(y):,} linhas"); del df, X
barra.fechar()
arqs_cache = sorted(PASTA_CACHE.glob("lote_*_de_*.npz")); assert len(arqs_cache) == B, f"o cache tem {len(arqs_cache)} lotes, esperava {B} (LOTE_LINHAS mudou entre execuções?)"
y_cache = np.concatenate([np.load(f)["y"] for f in arqs_cache]); assert len(y_cache) == N_TESTE and all(int((y_cache == COD[c]).sum()) == N_CLASSE[c] for c in CLASSES), "o cache do teste não bate com a view"
tam_cache = sum(f.stat().st_size for f in arqs_cache) / 2 ** 30
print(f"\n✔ Cache do teste: {len(y_cache):,} linhas em {B} lotes, {tam_cache:.2f} GiB em disco ({novos} lotes lidos agora do servidor) | {fmt_tempo(time.perf_counter() - t_ini)}")
T_SEC["2.1_cache"] = time.perf_counter() - t_ini


Cache do teste: concluído em 2min 33s

✔ Cache do teste: 7,216,487 linhas em 145 lotes, 0.18 GiB em disco (145 lotes lidos agora do servidor) | 2min 37s


## 3. Treino das sementes novas
Treina, no **refit do 05m/07m**, o padrão e o ajustado com cada semente nova (a 42 usa os modelos já gravados). Cada modelo é salvo ao ficar pronto. **Esta é a única parte do notebook que treina**, e só através de `treinar_semente`. Por padrão, treina também o ajustado com a semente 42 e confere que sai **idêntico** ao modelo do 07m (prova de que as sementes novas seguem o mesmo procedimento).

In [8]:
# ---------- 3.1 Treino das sementes novas (a única parte do notebook que treina modelos) ----------
t_ini = time.perf_counter(); PASTA_MOD = PASTA_EST / "modelos"; PASTA_MOD.mkdir(exist_ok=True)
def ordenar(df): return df.iloc[np.argsort(df["id_linha"].astype("uint64").to_numpy(), kind="stable")].reset_index(drop=True)
rf_df = ordenar(ler_view(V_REFIT)); X_rf, y_rf = rf_df[CM].to_numpy(float), rf_df["y"].to_numpy(int); assert np.isfinite(X_rf).all()
_rid = pd.read_csv(PASTA_PADRAO / "refit_ids.csv", dtype={"id_linha": str})
_m = rf_df[["id_linha", "y"]].merge(_rid[["id_linha", "y"]], on="id_linha", suffixes=("", "_05")); assert len(_m) == len(rf_df) == len(_rid) and bool((_m.y == _m.y_05).all()), "o refit atual não é o do notebook 05m (ids ou classes diferentes)"
print(f"Refit: {len(X_rf):,} linhas (as mesmas do 05m e do 07m; ids e classes conferidos), {len(CM)} atributos")

def treinar_semente(nome_cfg, semente):
    # Treina UM modelo (refit completo) com a semente dada e o salva em modelos/. Se já existir, reaproveita. É o ÚNICO lugar onde o treino é permitido.
    arq = PASTA_MOD / f"{nome_cfg}_semente{semente}.joblib"; info = PASTA_MOD / f"{nome_cfg}_semente{semente}.json"
    if arq.exists(): return arq, False
    hp = {**HP[nome_cfg], "random_state": semente}; TREINO_PERMITIDO[0] = True; silenciar_aviso_sklearn(); r0, c0 = time.perf_counter(), time.process_time()
    try: m = RandomForestClassifier(**hp).fit(X_rf, y_rf)
    finally: TREINO_PERMITIDO[0] = False
    dt = time.perf_counter() - r0; m.colunas_modelo_ = list(CM); tmp = tmp_novo(arq); joblib.dump(m, tmp, compress=COMPRESSAO_MODELO); os.replace(tmp, arq)
    salvar_json({"config": nome_cfg, "semente": semente, "hiperparametros": hp, "treino_relogio_s": dt, "nos_total": int(sum(e.tree_.node_count for e in m.estimators_)), "arquivo_mib": arq.stat().st_size / 2 ** 20, "gravado_em": datetime.now().isoformat(timespec="seconds")}, info)
    CRIADOS.append(f"modelos/{arq.name}"); return arq, True

NOVAS = [(c, s) for s in SEMENTES_ESTUDO if s != 42 for c in ("padrao", "ajustado")]; barra = Barra(len(NOVAS), "Treino das sementes novas"); treinados = 0
for c, s in NOVAS:
    barra.avancar(0, f"{ROT[c]} semente {s}: treinando..."); _, novo = treinar_semente(c, s); treinados += int(novo); barra.avancar(1, f"{ROT[c]} semente {s}: {'treinado agora' if novo else 'já existia'}")
barra.fechar()
REPRO = None
if VERIFICAR_REPRODUCAO:                                           # o procedimento deste notebook reproduz o do 07m? treina o ajustado com a semente 42 e compara com o modelo gravado (não é guardado)
    TREINO_PERMITIDO[0] = True; silenciar_aviso_sklearn()
    try: m_ref = RandomForestClassifier(**HP["ajustado"]).fit(X_rf, y_rf)
    finally: TREINO_PERMITIDO[0] = False
    m_grav = joblib.load(ARQ_42["ajustado"]); dif = float(np.abs(m_ref.predict_proba(X_rf[:20000]) - m_grav.predict_proba(X_rf[:20000])).max()); REPRO = {"diferenca_maxima": dif, "nos_refeito": int(sum(e.tree_.node_count for e in m_ref.estimators_)), "nos_gravado": int(sum(e.tree_.node_count for e in m_grav.estimators_))}
    print(f"{'✔' if dif < 1e-9 else '✘'} Reprodução: o ajustado semente 42 retreinado aqui {'é IDÊNTICO ao' if dif < 1e-9 else 'DIFERE do'} modelo gravado pelo 07m (diferença máxima nas probabilidades: {dif:.1e}; nós {REPRO['nos_refeito']:,} × {REPRO['nos_gravado']:,})"); del m_ref, m_grav
tt = pd.DataFrame([json.loads(p.read_text(encoding="utf-8")) for p in sorted(PASTA_MOD.glob("*.json"))]); TEMPOS_TREINO = tt
if len(tt): display(tt[["config", "semente", "treino_relogio_s", "nos_total", "arquivo_mib"]].round(2).sort_values(["config", "semente"]).reset_index(drop=True))
print(f"✔ {len(NOVAS)} modelos novos prontos ({treinados} treinados agora) | {fmt_tempo(time.perf_counter() - t_ini)}"); T_SEC["3.1_treino"] = time.perf_counter() - t_ini


Refit: 299,589 linhas (as mesmas do 05m e do 07m; ids e classes conferidos), 44 atributos


Treino das sementes novas: concluído em 7min 30s
✔ Reprodução: o ajustado semente 42 retreinado aqui é IDÊNTICO ao modelo gravado pelo 07m (diferença máxima nas probabilidades: 3.3e-16; nós 1,267,902 × 1,267,902)


,config,semente,treino_relogio_s,nos_total,arquivo_mib
0,ajustado,43,48.85,1265714,27.72
1,ajustado,44,48.11,1262964,27.66
2,ajustado,45,50.03,1262884,27.64
3,ajustado,46,49.38,1278104,27.99
4,padrao,43,55.73,4311468,67.41
5,padrao,44,53.51,4297026,67.23
6,padrao,45,53.50,4298722,67.22
7,padrao,46,53.78,4305430,67.33


✔ 8 modelos novos prontos (8 treinados agora) | 8min 43s


## 4. Previsão no teste e métricas
Para cada modelo (um por vez, para poupar memória): carrega, prevê todos os lotes do cache e calcula as métricas do 06m/08m, o desempenho ataque × BENIGN e o destino do `Syn`. O resultado de cada modelo é salvo em `resultados/`; se existir, é reaproveitado.

In [9]:
# ---------- 4.1 Previsão no teste e métricas, um modelo por vez ----------
t_ini = time.perf_counter(); PASTA_RES = PASTA_EST / "resultados"; PASTA_RES.mkdir(exist_ok=True); I_SYN = COD["Syn"]
def carregar_modelo(nome_cfg, semente):
    m = joblib.load(ARQ_42[nome_cfg] if semente == 42 else PASTA_MOD / f"{nome_cfg}_semente{semente}.joblib")
    assert all(getattr(m, k) == v for k, v in {**HP[nome_cfg], "random_state": semente}.items()) and list(m.classes_) == list(range(K)) and list(m.colunas_modelo_) == CM, f"modelo {nome_cfg} semente {semente} não corresponde aos hiperparâmetros e atributos esperados"
    return m
def avaliar(nome_cfg, semente):
    arq = PASTA_RES / f"{nome_cfg}_semente{semente}.json"
    if arq.exists(): return json.loads(arq.read_text(encoding="utf-8")), False
    m = carregar_modelo(nome_cfg, semente); P = np.empty((N_TESTE, K), np.float32); y = np.empty(N_TESTE, np.uint8); pos = 0; t_prev = 0.0
    for f in arqs_cache:
        d = np.load(f); X = d["X"]; n = len(X)
        if n == 0: continue                                          # faixa de ids sem linhas
        silenciar_aviso_sklearn(); t0 = time.perf_counter(); P[pos:pos + n] = m.predict_proba(X); t_prev += time.perf_counter() - t0; y[pos:pos + n] = d["y"]; pos += n
    assert pos == N_TESTE and np.array_equal(y, y_cache)
    esc, arr = metricas_multi_grande(y, P); cm = arr["cm"]; i_esp = COD.get("UDPLag"); col = colapsa(cm, I_SYN); sy = cm[I_SYN] / cm[I_SYN].sum()
    res = {"config": nome_cfg, "semente": semente, "metricas": esc, "f1_macro_sem_UDPLag": float(np.delete(arr["f1"], i_esp).mean()) if i_esp is not None else None, "recall": {c: float(arr["recall"][COD[c]]) for c in CLASSES},
           "precisao": {c: float(arr["precisao"][COD[c]]) for c in CLASSES}, "f1": {c: float(arr["f1"][COD[c]]) for c in CLASSES}, "roc_auc_classe": {c: float(arr["auc"][COD[c]]) for c in CLASSES}, "ataque_vs_benign": col,
           "syn_para": {c: float(sy[COD[c]]) for c in CLASSES}, "matriz_confusao": cm.astype(int).tolist(), "tempo_previsao_s": t_prev, "vazao_fluxos_s": N_TESTE / t_prev, "gravado_em": datetime.now().isoformat(timespec="seconds")}
    salvar_json(res, arq); CRIADOS.append(f"resultados/{arq.name}"); del m, P, y; gc.collect(); return res, True
ORDEM = [(c, s) for s in SEMENTES_ESTUDO for c in ("padrao", "ajustado")]; barra = Barra(len(ORDEM), "Previsão e métricas por modelo"); RESULTADOS = {}; novos = 0
for c, s in ORDEM:
    barra.avancar(0, f"{ROT[c]} semente {s}: prevendo o teste... (memória {rss_mib():,.0f} MiB)"); r, novo = avaliar(c, s); RESULTADOS[(c, s)] = r; novos += int(novo)
    barra.avancar(1, f"{ROT[c]} semente {s}: F1 macro {r['metricas']['f1_macro']:.4f}, MCC {r['metricas']['mcc']:.4f}" + ("" if novo else " (já pronto)"))
barra.fechar(); print(f"\n✔ {len(RESULTADOS)} modelos avaliados no teste ({novos} calculados agora) | {fmt_tempo(time.perf_counter() - t_ini)}"); T_SEC["4.1_previsao"] = time.perf_counter() - t_ini


Previsão e métricas por modelo: concluído em 12min 46s

✔ 10 modelos avaliados no teste (10 calculados agora) | 12min 46s


## 5. Resultados por semente
Tabelas por modelo e semente, o resumo (média, desvio, mínimo e máximo), a **diferença pareada** (ajustado − padrão na mesma semente), se as **faixas das sementes se sobrepõem** e o gráfico. É descritivo: o teste tem uma só partição e muitas linhas repetidas, e as sementes medem só o ruído do sorteio do modelo.

In [10]:
# ---------- 5.1 Resultados por semente ----------
def linha(r):
    m, col, rc = r["metricas"], r["ataque_vs_benign"], r["recall"]
    return {"config": r["config"], "semente": r["semente"], "roc_auc": m["roc_auc_ovr_macro"], "acuracia_balanceada": m["acuracia_balanceada"], "acuracia": m["acuracia"], "f1_macro": m["f1_macro"], "f1_macro_sem_UDPLag": r["f1_macro_sem_UDPLag"], "mcc": m["mcc"],
            **{f"recall_{c}": rc[c] for c in CLASSES}, "ataque_recall": col["recall dos ataques"], "ataque_falso_alarme": col["falso alarme"], "ataque_mcc": col["MCC ataque × BENIGN"], "syn_para_BENIGN_%": 100 * r["syn_para"]["BENIGN"],
            "syn_para_UDPLag_%": 100 * r["syn_para"]["UDPLag"], "MSSQL_para_NetBIOS_%": 100 * r["matriz_confusao"][COD["MSSQL"]][COD["NetBIOS"]] / sum(r["matriz_confusao"][COD["MSSQL"]]), "UDP_para_UDPLag_%": 100 * r["matriz_confusao"][COD["UDP"]][COD["UDPLag"]] / sum(r["matriz_confusao"][COD["UDP"]])}
RES = pd.DataFrame([linha(RESULTADOS[k]) for k in ORDEM]); MET = [c for c in RES.columns if c not in ("config", "semente")]
CHAVES = ["roc_auc", "acuracia_balanceada", "f1_macro", "f1_macro_sem_UDPLag", "mcc", "recall_Syn", "recall_MSSQL", "recall_UDP", "recall_UDPLag", "ataque_recall", "ataque_falso_alarme", "syn_para_BENIGN_%", "syn_para_UDPLag_%"]
print("== Resultados por modelo e semente (teste: 7,2 milhões de linhas) =="); display(RES[["config", "semente"] + CHAVES].round(4))
resumo = RES.groupby("config")[MET].agg(["mean", "std", "min", "max"]); resumo = resumo.reindex(["padrao", "ajustado"]); RESUMO = pd.DataFrame({(m, est): resumo[m][est] for m in MET for est in ("mean", "std", "min", "max")}).T
RESUMO.index = pd.MultiIndex.from_tuples(RESUMO.index, names=["métrica", "estatística"]); print("\n== Resumo por modelo (média, desvio entre sementes, mínimo e máximo) =="); display(RESUMO.loc[CHAVES].round(4))
pad, aju = RES[RES.config == "padrao"].set_index("semente")[MET], RES[RES.config == "ajustado"].set_index("semente")[MET]; D = (aju - pad).loc[list(SEMENTES_ESTUDO)]
MELHOR_MAIOR = {m: (m not in ("ataque_falso_alarme", "syn_para_BENIGN_%", "MSSQL_para_NetBIOS_%", "UDP_para_UDPLag_%", "syn_para_UDPLag_%")) for m in MET}
par = pd.DataFrame({"Δ médio (ajustado − padrão)": D.mean(), "desvio do Δ": D.std(ddof=1), "Δ mínimo": D.min(), "Δ máximo": D.max(), "sementes em que o ajustado é melhor": [int(((D[m] > 0) if MELHOR_MAIOR[m] else (D[m] < 0)).sum()) for m in MET],
                    "faixas das sementes se sobrepõem?": ["sim" if (aju[m].min() <= pad[m].max() and pad[m].min() <= aju[m].max()) else "NÃO" for m in MET]}, index=MET)
PAR = par; print(f"\n== Diferença pareada por semente (ajustado − padrão; {len(SEMENTES_ESTUDO)} sementes) =="); display(par.loc[CHAVES].round(4))
print("\n== Leitura (descritiva) ==")
for m in ("f1_macro", "mcc", "roc_auc", "recall_Syn", "ataque_recall", "syn_para_BENIGN_%"):
    p_, a_ = pad[m], aju[m]; print(f"  {m:20s} padrão {p_.min():.4f}–{p_.max():.4f} (média {p_.mean():.4f}) | ajustado {a_.min():.4f}–{a_.max():.4f} (média {a_.mean():.4f}) | Δ médio {D[m].mean():+.4f} | faixas se sobrepõem: {par.loc[m, 'faixas das sementes se sobrepõem?']} | ajustado melhor em {par.loc[m, 'sementes em que o ajustado é melhor']}/{len(SEMENTES_ESTUDO)}")
ref = None
try:
    gr = pd.read_csv(PASTA_RUN / "resultados_grade.csv"); pc_ = gr[(gr.n_estimators == HP["ajustado"]["n_estimators"]) & (gr.profundidade.astype(str) == "None") & (gr.min_samples_leaf.isin([HP["ajustado"]["min_samples_leaf"], HP["padrao"]["min_samples_leaf"]]))]
    ref = pc_[["min_samples_leaf", "cv_mcc", "cv_mcc_dp_semente", "cv_roc_auc", "cv_roc_auc_dp_semente"]]; print("\nReferência: desvio entre sementes na VALIDAÇÃO CRUZADA da busca (07m):"); display(ref.round(5))
except Exception as e: print("(referência da validação cruzada não disponível:", type(e).__name__, ")")
try:
    import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt
    fig, axs = plt.subplots(2, 3, figsize=(13, 7)); pan = [("f1_macro", "F1 macro"), ("mcc", "MCC"), ("roc_auc", "ROC-AUC um-contra-resto"), ("recall_Syn", "recall do Syn"), ("ataque_recall", "recall dos ataques (ataque × BENIGN)"), ("syn_para_BENIGN_%", "Syn previsto como BENIGN (%)")]
    for ax, (m, tit) in zip(axs.ravel(), pan):
        for c, cor in (("padrao", "tab:blue"), ("ajustado", "tab:orange")):
            d_ = RES[RES.config == c].sort_values("semente"); ax.plot(np.full(len(d_), 0 if c == "padrao" else 1) + np.linspace(-0.12, 0.12, len(d_)), d_[m], "o", color=cor, label=ROT[c])
        for s_ in SEMENTES_ESTUDO: ax.plot([0, 1], [pad.loc[s_, m], aju.loc[s_, m]], "-", color="gray", alpha=0.4, lw=0.8)
        ax.set_xticks([0, 1]); ax.set_xticklabels(["padrão", "ajustado"]); ax.set_title(tit, fontsize=10); ax.grid(axis="y", alpha=.3)
    fig.suptitle(f"Estudo de sementes no teste ({len(SEMENTES_ESTUDO)} sementes; cada linha cinza liga a mesma semente)"); fig.tight_layout(); gravar("grafico_sementes.png", lambda p_: fig.savefig(p_, dpi=130)); plt.close(fig)
except ImportError: print("(matplotlib não está instalado: gráfico não gerado)")


== Resultados por modelo e semente (teste: 7,2 milhões de linhas) ==


,config,semente,roc_auc,acuracia_balanceada,f1_macro,f1_macro_sem_UDPLag,mcc,recall_Syn,recall_MSSQL,recall_UDP,recall_UDPLag,ataque_recall,ataque_falso_alarme,syn_para_BENIGN_%,syn_para_UDPLag_%
0,padrao,42,0.9133,0.6908,0.5626,0.6564,0.4324,0.2445,0.3681,0.8744,0.4039,0.9994,0.0176,0.0745,75.4008
1,ajustado,42,0.9248,0.6771,0.4148,0.4839,0.3688,0.1617,0.3411,0.9081,0.3573,0.7255,0.0194,40.2092,43.5560
2,padrao,43,0.9130,0.6894,0.5452,0.6360,0.4300,0.2389,0.3643,0.8753,0.4022,0.9973,0.0171,0.3816,75.6653
3,ajustado,43,0.9260,0.6718,0.4094,0.4776,0.3626,0.1416,0.3423,0.9066,0.3411,0.6758,0.0213,47.4857,38.2922
4,padrao,44,0.9072,0.6806,0.4219,0.4922,0.3722,0.1863,0.3646,0.8753,0.3950,0.7222,0.0195,40.6896,40.6194
5,ajustado,44,0.9247,0.6740,0.4137,0.4826,0.3674,0.1573,0.3410,0.9085,0.3393,0.6761,0.0205,47.4465,36.7546
6,padrao,45,0.9151,0.6775,0.4183,0.4880,0.3666,0.1668,0.3654,0.8750,0.3914,0.7210,0.0180,40.8618,42.3960
7,ajustado,45,0.9256,0.6730,0.4085,0.4766,0.3616,0.1366,0.3413,0.9097,0.3519,0.6753,0.0198,47.5667,38.7039
8,padrao,46,0.9196,0.6832,0.4689,0.5470,0.4097,0.1780,0.3639,0.8738,0.4219,0.9766,0.0178,3.4198,78.7136
9,ajustado,46,0.9224,0.6697,0.4117,0.4803,0.3652,0.1555,0.3393,0.9020,0.3196,0.6764,0.0207,47.3935,36.9936



== Resumo por modelo (média, desvio entre sementes, mínimo e máximo) ==


config                            padrao  ajustado
métrica             estatística                   
roc_auc             mean          0.9136    0.9247
                    std           0.0045    0.0014
                    min           0.9072    0.9224
                    max           0.9196    0.9260
acuracia_balanceada mean          0.6843    0.6731
                    std           0.0057    0.0027
                    min           0.6775    0.6697
                    max           0.6908    0.6771
f1_macro            mean          0.4834    0.4116
                    std           0.0677    0.0027
                    min           0.4183    0.4085
                    max           0.5626    0.4148
f1_macro_sem_UDPLag mean          0.5639    0.4802
                    std           0.0790    0.0031
                    min           0.4880    0.4766
                    max           0.6564    0.4839
mcc                 mean          0.4022    0.3651
                    std           0.0313    0.0031
                    min           0.3666    0.3616
                    max           0.4324    0.3688
recall_Syn          mean          0.2029    0.1505
                    std           0.0362    0.0108
                    min           0.1668    0.1366
                    max           0.2445    0.1617
recall_MSSQL        mean          0.3653    0.3410
                    std           0.0017    0.0011
                    min           0.3639    0.3393
                    max           0.3681    0.3423
recall_UDP          mean          0.8748    0.9070
                    std           0.0007    0.0030
                    min           0.8738    0.9020
                    max           0.8753    0.9097
recall_UDPLag       mean          0.4029    0.3418
                    std           0.0118    0.0145
                    min           0.3914    0.3196
                    max           0.4219    0.3573
ataque_recall       mean          0.8833    0.6858
                    std           0.1479    0.0222
                    min           0.7210    0.6753
                    max           0.9994    0.7255
ataque_falso_alarme mean          0.0180    0.0203
                    std           0.0009    0.0008
                    min           0.0171    0.0194
                    max           0.0195    0.0213
syn_para_BENIGN_%   mean         17.0855   46.0203
                    std          21.6657    3.2491
                    min           0.0745   40.2092
                    max          40.8618   47.5667
syn_para_UDPLag_%   mean         62.5590   38.8601
                    std          19.2714    2.7530
                    min          40.6194   36.7546
                    max          78.7136   43.5560


== Diferença pareada por semente (ajustado − padrão; 5 sementes) ==


,Δ médio (ajustado − padrão),desvio do Δ,Δ mínimo,Δ máximo,sementes em que o ajustado é melhor,faixas das sementes se sobrepõem?
roc_auc,0.0111,0.0054,0.0028,0.0175,5,NÃO
acuracia_balanceada,-0.0111,0.0054,-0.0176,-0.0045,0,NÃO
f1_macro,-0.0717,0.0671,-0.1479,-0.0082,0,NÃO
f1_macro_sem_UDPLag,-0.0837,0.0782,-0.1725,-0.0096,0,NÃO
mcc,-0.0371,0.0307,-0.0674,-0.0047,0,sim
recall_Syn,-0.0524,0.0349,-0.0974,-0.0226,0,NÃO
recall_MSSQL,-0.0243,0.0018,-0.0270,-0.0219,0,NÃO
recall_UDP,0.0322,0.0026,0.0282,0.0347,5,NÃO
recall_UDPLag,-0.0610,0.0245,-0.1023,-0.0395,0,NÃO
ataque_recall,-0.1975,0.1394,-0.3215,-0.0458,0,sim



== Leitura (descritiva) ==
  f1_macro             padrão 0.4183–0.5626 (média 0.4834) | ajustado 0.4085–0.4148 (média 0.4116) | Δ médio -0.0717 | faixas se sobrepõem: NÃO | ajustado melhor em 0/5
  mcc                  padrão 0.3666–0.4324 (média 0.4022) | ajustado 0.3616–0.3688 (média 0.3651) | Δ médio -0.0371 | faixas se sobrepõem: sim | ajustado melhor em 0/5
  roc_auc              padrão 0.9072–0.9196 (média 0.9136) | ajustado 0.9224–0.9260 (média 0.9247) | Δ médio +0.0111 | faixas se sobrepõem: NÃO | ajustado melhor em 5/5
  recall_Syn           padrão 0.1668–0.2445 (média 0.2029) | ajustado 0.1366–0.1617 (média 0.1505) | Δ médio -0.0524 | faixas se sobrepõem: NÃO | ajustado melhor em 0/5
  ataque_recall        padrão 0.7210–0.9994 (média 0.8833) | ajustado 0.6753–0.7255 (média 0.6858) | Δ médio -0.1975 | faixas se sobrepõem: sim | ajustado melhor em 0/5
  syn_para_BENIGN_%    padrão 0.0745–40.8618 (média 17.0855) | ajustado 40.2092–47.5667 (média 46.0203) | Δ médio +28.9349 | fa

,min_samples_leaf,cv_mcc,cv_mcc_dp_semente,cv_roc_auc,cv_roc_auc_dp_semente


## 6. Gravação e conferências
Grava os CSVs, o JSON e o gráfico (**só arquivos novos**) e confere: que todos os modelos têm resultado, que a **semente 42 reproduz exatamente** o 06m e o 08m, que o treino bloqueado nunca foi acionado fora de `treinar_semente`, e que **nenhum arquivo pré-existente** das pastas do 07m e do 05m foi alterado.

In [11]:
# ---------- 6.1 Gravação e conferências ----------
gravar_csv(RES, "resultados_por_semente.csv", index=False); gravar_csv(RESUMO.reset_index(), "resumo_por_modelo.csv", index=False); gravar_csv(PAR.reset_index(names="metrica"), "diferenca_pareada.csv", index=False)
gravar_csv(D.reset_index(), "diferenca_por_semente.csv", index=False); gravar_csv(TEMPOS_TREINO, "tempos_treino_modelos.csv", index=False)
gravar_json({"sementes": SEMENTES_ESTUDO, "modelos_avaliados": len(RESULTADOS), "padrao": {"pasta": PASTA_PADRAO.name, "hiperparametros": HP["padrao"]}, "ajustado": {"pasta": PASTA_RUN.name, "hiperparametros": HP["ajustado"]},
             "n_teste": N_TESTE, "reproducao_ajustado_42": REPRO, "tempos_etapas_s": T_SEC, "data": datetime.now().isoformat(timespec="seconds")}, "estudo_sementes.json")
ok = []
def verifica(nome, cond, detalhe=""):
    ok.append(bool(cond)); print(("✔" if cond else "✘"), nome, detalhe)
cm_de = lambda r: np.array(r["matriz_confusao"])
verifica("Todas as combinações (2 modelos × cada semente) têm resultado", len(RESULTADOS) == 2 * len(SEMENTES_ESTUDO) and all((PASTA_RES / f"{c}_semente{s}.json").exists() for c, s in ORDEM), f"({len(RESULTADOS)})")
verifica("As matrizes de confusão fecham com o total e com as linhas por classe, em todos os modelos", all(int(cm_de(r).sum()) == N_TESTE and all(int(cm_de(r)[COD[c]].sum()) == N_CLASSE[c] for c in CLASSES) for r in RESULTADOS.values()))
ref_pad = json.loads((PASTA_PADRAO / "metricas_teste.json").read_text(encoding="utf-8")); ref_aj = json.loads((PASTA_RUN / "metricas_teste.json").read_text(encoding="utf-8"))
cm_ref = lambda ref: np.array([[ref["matriz_confusao"][f"real {a}"][f"prev. {b}"] for b in CLASSES] for a in CLASSES])
verifica("Semente 42 do PADRÃO reproduz exatamente a matriz de confusão do 06m", np.array_equal(cm_de(RESULTADOS[("padrao", 42)]), cm_ref(ref_pad)))
verifica("Semente 42 do AJUSTADO reproduz exatamente a matriz de confusão do 08m", np.array_equal(cm_de(RESULTADOS[("ajustado", 42)]), cm_ref(ref_aj)))
verifica("Semente 42: as métricas coincidem com as do 06m e do 08m", abs(RESULTADOS[("padrao", 42)]["metricas"]["f1_macro"] - ref_pad["metricas"]["f1_macro"]) < 1e-9 and abs(RESULTADOS[("ajustado", 42)]["metricas"]["mcc"] - ref_aj["metricas"]["mcc"]) < 1e-9)
if REPRO is not None: verifica("O ajustado semente 42 retreinado aqui é idêntico ao modelo gravado pelo 07m (mesmo procedimento de treino)", REPRO["diferenca_maxima"] < 1e-9, f"(diferença {REPRO['diferenca_maxima']:.1e})")
verifica("Modelos novos: hiperparâmetros certos, semente certa, classes 0..K-1 e atributos na ordem do pré-processamento", all(((lambda m: all(getattr(m, k) == v for k, v in {**HP[c], "random_state": s}.items()) and list(m.classes_) == list(range(K)) and list(m.colunas_modelo_) == CM)(joblib.load(PASTA_MOD / f"{c}_semente{s}.joblib"))) for c in ("padrao", "ajustado") for s in SEMENTES_ESTUDO if s != 42))
verifica("Cada semente nova produziu um modelo diferente (nós totais distintos entre sementes)", all(TEMPOS_TREINO[TEMPOS_TREINO.config == c].nos_total.nunique() == len(TEMPOS_TREINO[TEMPOS_TREINO.config == c]) for c in ("padrao", "ajustado")) if len(SEMENTES_ESTUDO) > 2 else True)
verifica("Cache do teste: todas as linhas e as classes por linha", len(y_cache) == N_TESTE)
verifica("Probabilidades: previsões de 1 lote somam 1 (modelo ajustado semente 42)", (lambda m, X: bool(np.allclose(m.predict_proba(X).sum(axis=1), 1.0, atol=1e-5)))(carregar_modelo("ajustado", 42), next(np.load(f)["X"][:2000] for f in arqs_cache if len(np.load(f)["y"]))))
_depois = {("busca/" + str(f.relative_to(PASTA_RUN))): (f.stat().st_size, f.stat().st_mtime_ns) for f in PASTA_RUN.rglob("*") if f.is_file() and PASTA_EST not in f.parents}; _depois.update({("padrao/" + str(f.relative_to(PASTA_PADRAO))): (f.stat().st_size, f.stat().st_mtime_ns) for f in PASTA_PADRAO.rglob("*") if f.is_file()})
_mexidos = [n for n, v in SNAP_ANTES.items() if _depois.get(n) != v]
verifica(f"Nenhum dos {len(SNAP_ANTES)} arquivos que já existiam (pastas do 07m/08m e do 05m/06m) foi alterado, sobrescrito ou apagado", not _mexidos, f"({_mexidos[:3]})" if _mexidos else "")
verifica("O treino só aconteceu dentro de treinar_semente (nenhuma tentativa bloqueada)", len(TREINOS_BLOQUEADOS) == 0)
verifica("Arquivos esperados gravados", all((PASTA_EST / f).exists() for f in ("resultados_por_semente.csv", "resumo_por_modelo.csv", "diferenca_pareada.csv", "estudo_sementes.json", "tempos_treino_modelos.csv")))
assert all(ok), "há checagens reprovadas acima"
print(f"\n✔ {sum(ok)}/{len(ok)} checagens aprovadas. Resultados em {PASTA_EST}"); print(f"\n⏱ Tempo total do notebook (etapas medidas): {fmt_tempo(float(sum(T_SEC.values())))}")
print(f"\nArquivos NOVOS criados nesta execução ({len(CRIADOS)}): {', '.join(CRIADOS[:12])}{' ...' if len(CRIADOS) > 12 else ''}")
if MANTIDOS: print(f"Arquivos que já existiam e foram MANTIDOS ({len(MANTIDOS)}): {', '.join(MANTIDOS[:12])}{' ...' if len(MANTIDOS) > 12 else ''}")
print("\nPróximo passo: me mande o texto desta saída (e, se quiser, resultados_por_semente.csv, diferenca_pareada.csv e grafico_sementes.png).")


✔ Todas as combinações (2 modelos × cada semente) têm resultado (10)
✔ As matrizes de confusão fecham com o total e com as linhas por classe, em todos os modelos 
✔ Semente 42 do PADRÃO reproduz exatamente a matriz de confusão do 06m 
✔ Semente 42 do AJUSTADO reproduz exatamente a matriz de confusão do 08m 
✔ Semente 42: as métricas coincidem com as do 06m e do 08m 
✔ O ajustado semente 42 retreinado aqui é idêntico ao modelo gravado pelo 07m (mesmo procedimento de treino) (diferença 3.3e-16)
✔ Modelos novos: hiperparâmetros certos, semente certa, classes 0..K-1 e atributos na ordem do pré-processamento 
✔ Cada semente nova produziu um modelo diferente (nós totais distintos entre sementes) 
✔ Cache do teste: todas as linhas e as classes por linha 
✔ Probabilidades: previsões de 1 lote somam 1 (modelo ajustado semente 42) 
✔ Nenhum dos 396 arquivos que já existiam (pastas do 07m/08m e do 05m/06m) foi alterado, sobrescrito ou apagado 
✔ O treino só aconteceu dentro de treinar_semente (ne

In [12]:
import pandas as pd
gr = pd.read_csv(r"C:\Users\Bruno\downloads\tcc_codigo\railway-ch\artefatos_modelos\rf_multiclasse_busca\rfm_busca_20261102_001711\resultados_grade.csv", keep_default_na=False)
print(gr[(gr.n_estimators == 100) & (gr.profundidade == "None") & gr.min_samples_leaf.isin([1, 5])][["min_samples_leaf", "cv_mcc", "cv_mcc_dp_semente", "cv_roc_auc", "cv_roc_auc_dp_semente"]])

   min_samples_leaf    cv_mcc  cv_mcc_dp_semente  cv_roc_auc  cv_roc_auc_dp_semente
6                 1  0.898737           0.000224    0.990467               0.000054
7                 5  0.914107           0.000093    0.994894               0.000014
